# Γ1.c F1 — lr_cr sweep (diagnostic, smoke n=3)

Per the F1 precommit at [notes/notes/2026-05-27-path-gamma-gamma1-f1-lr-cr-sweep-precommit.md](https://github.com/). Diagnostic-only — no graduation claim under any outcome.

**Design.** 5 lr_cr values × 3 seeds = 15 parallel CUDA subprocesses. All Γ1.c. PathC baseline at seeds 0..2 is loaded from the Γ1 headline gate's Drive output (not re-run).

| lr_cr | flag | purpose |
|---|---|---|
| 0.01 | `--lr-cr 0.01` | weak end — does Γ1.c saturate near zero? |
| 0.05 | `--lr-cr 0.05` | mid-low |
| 0.1 | `--lr-cr 0.1` | headline-equivalent (Report 113 reproducibility check at smoke n=3) |
| 0.2 | `--lr-cr 0.2` | mid-high — closer to where atom-pair magnitudes would need a boost |
| 0.5 | `--lr-cr 0.5` | strong end — does PathC-magnitude effect appear? |

**Hypothesis test (per precommit §"Falsifiable diagnostic criterion"):**

| hypothesis | empirical signature | next step |
|---|---|---|
| (a) effective-lr mismatch | max(lr_cr=0.2 or 0.5) per-seed mean Δ ≳ 0.04 (≈ within 30% of PathC's +0.055) | new precommit: full n=10 headline at winning lr_cr |
| (b) atom-vs-atom geometry carries less signal | max across all lr_cr per-seed mean Δ ≲ 0.02 | Γ1 family closes; Γ2 (bundle-first) or Γ3 (SFA-head) precommit |
| inconclusive | curve non-monotonic / noisy at n=3 | extend at n=5 / n=10 at most promising lr_cr |

**Pre-flight.** Same as the headline gate. The verify cell fails fast if the Γ1 implementation isn't on remote.

In [ ]:
# 1. Clone the repo + verify Γ1 implementation is present on the branch.
import subprocess, sys, os
from pathlib import Path
REPO_DIR = '/content/Neuro-AI'
BRANCH = 'codex/phase5-prime-bundle-first-scene-memory'
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout {BRANCH}
!git log --oneline -3
os.chdir(REPO_DIR)
online_cb = Path(REPO_DIR) / 'src' / 'energy_memory' / 'phase34' / 'online_codebook.py'
src = online_cb.read_text()
if 'use_context_residual' not in src or '_apply_context_residual' not in src:
    raise SystemExit('Γ1 implementation not on branch — push and re-run.')
driver_src = (Path(REPO_DIR) / 'experiments' / 'c3_phase3_exit_criterion.py').read_text()
if '--lr-cr' not in driver_src or '--use-context-residual' not in driver_src:
    raise SystemExit('--lr-cr or --use-context-residual flag not in driver. Push and re-run.')
print('Γ1 implementation verified on branch.')
print(f'current HEAD: {subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()}')

In [ ]:
# 2. Mount Drive (also needed to load PathC baseline from headline-gate output).
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuro-ai/results', exist_ok=True)
# Verify the headline-gate PathC baseline at seeds 0..2 exists on Drive.
from pathlib import Path
missing = []
for seed in (0, 1, 2):
    p = Path(f'/content/drive/MyDrive/neuro-ai/results/gamma1_headline_2026-05-27/pathc_baseline_seed{seed}/c3_summary.json')
    if not p.exists():
        missing.append(str(p))
if missing:
    print('⚠ PathC baseline JSONs missing on Drive (F1 still runs, but per-seed paired comparison will be partial):')
    for m in missing:
        print(f'  - {m}')
else:
    print('PathC baseline at seeds 0..2 found on Drive ✓')

In [ ]:
# 3. Install deps.
!pip install -q torch datasets huggingface_hub

In [ ]:
# 4. Pre-warm WikiText-2 HF cache. Parent CPU-only.
import sys; sys.path.insert(0, '/content/Neuro-AI/src')
from datasets import load_dataset
_ = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train[:1%]')
print('WikiText-2 cache pre-warmed.')

In [ ]:
# 5. GPU info.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 6. SMOKE — one tiny Γ1 subprocess to verify the runtime + CLI path.
import subprocess, sys, os, json
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
smoke_out = Path('reports/f1_smoke_2026-05-27')
smoke_out.mkdir(parents=True, exist_ok=True)
smoke_log = Path('reports/f1_smoke.log')
cmd = [sys.executable, 'experiments/c3_phase3_exit_criterion.py',
       '--seeds', '0', '--device', 'cuda',
       '--use-context-residual', '--no-pull-push', '--lr-cr', '0.1',
       '--D', '128', '--landscape-size', '8',
       '--n-consolidation-events', '50',
       '--vocab-size', '50', '--K', '3', '--beta', '10',
       '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
       '--corpus-source', 'synthetic',
       '--output-dir', str(smoke_out)]
with smoke_log.open('w') as logf:
    rc = subprocess.call(cmd, stdout=logf, stderr=subprocess.STDOUT)
print(f'smoke exit code: {rc}; json exists: {(smoke_out / "c3_summary.json").exists()}')
if rc != 0:
    !tail -30 {smoke_log}
    raise SystemExit('F1 smoke failed — abort.')
with open(smoke_out / 'c3_summary.json') as f:
    sm = json.load(f)
header = sm.get('header', {})
if not header.get('use_context_residual') or header.get('use_pull_push'):
    raise SystemExit(
        f'Flag misroute: header use_context_residual='
        f'{header.get("use_context_residual")} use_pull_push='
        f'{header.get("use_pull_push")}; expected True, False'
    )
print('F1 smoke OK; flags propagated correctly.')

In [ ]:
# 7. PARALLEL launch — 5 lr_cr values × 3 seeds = 15 per-seed subprocesses.
import subprocess, os, time, signal, sys
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
PY = sys.executable

# Operating point per the F1 precommit §"Operating point" — identical
# to the Γ1 headline gate in every dimension except lr_cr.
BASE_FLAGS = [
    '--device', 'cuda',
    '--corpus-source', 'wikitext',
    '--wikitext-name', 'wikitext-2-raw-v1',
    '--vocab-cap', '1000',
    '--window', '8',
    '--D', '4096',
    '--landscape-size', '64',
    '--beta', '10',
    '--K', '5',
    '--n-consolidation-events', '1000',
    '--alpha-anti', '0.01',
    '--repulsion-step-size', '0.05',
    '--lr-pull', '0.1',
    '--lr-push', '0.05',
    '--use-context-residual', '--no-pull-push',
]

LR_CR_VALUES = [0.01, 0.05, 0.1, 0.2, 0.5]
SEEDS = [0, 1, 2]

ENTRIES = [(lr, seed) for lr in LR_CR_VALUES for seed in SEEDS]
print(f'launching {len(ENTRIES)} subprocesses '
      f'({len(LR_CR_VALUES)} lr_cr × {len(SEEDS)} seeds)')

log_root = Path('reports/f1_logs')
log_root.mkdir(parents=True, exist_ok=True)

def out_dir_for(lr, seed):
    return f'reports/f1_lr{lr}_seed{seed}_2026-05-27'

def launch(lr_cr, seed):
    out_dir = out_dir_for(lr_cr, seed)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'lr{lr_cr}_seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [PY, 'experiments/c3_phase3_exit_criterion.py',
           '--seeds', str(seed),
           *BASE_FLAGS,
           '--lr-cr', str(lr_cr),
           '--output-dir', out_dir]
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
    return proc, logf, out_dir, log_path

def snapshot(remaining, total, t0):
    elapsed = (time.time() - t0) / 60
    n_done = total - len(remaining)
    print(f'  --- snapshot at {elapsed:.1f} min — {n_done}/{total} done, '
          f'{len(remaining)} running ---')
    try:
        gpu = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        print(f'  GPU: {gpu}')
    except Exception as e:
        print(f'  GPU snapshot failed: {e}')
    from collections import Counter
    by_lr = Counter()
    for key in remaining:
        lr = key.split('_seed')[0].replace('lr', '')
        by_lr[lr] += 1
    for lr in LR_CR_VALUES:
        n_run = by_lr.get(str(lr), 0)
        print(f'    lr_cr={lr}: {len(SEEDS) - n_run}/{len(SEEDS)} done')

procs = {}
for lr_cr, seed in ENTRIES:
    key = f'lr{lr_cr}_seed{seed}'
    procs[key] = launch(lr_cr, seed)
    time.sleep(1.5)
print(f'all {len(procs)} cells launched  ({time.strftime("%H:%M:%S")})')

t0 = time.time()
remaining = dict(procs)
total = len(procs)
failures = []
poll_count = 0
try:
    while remaining:
        done_this_round = []
        for key, (proc, logf, out_dir, log_path) in remaining.items():
            rc = proc.poll()
            if rc is not None:
                logf.close()
                elapsed = (time.time() - t0) / 60
                json_exists = Path(out_dir, 'c3_summary.json').exists()
                ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                print(f'  [{elapsed:5.1f} min] {key:>25}: {ok}  json={json_exists}')
                if rc != 0:
                    failures.append(key)
                    print(f'    --- last 30 lines of {log_path} ---')
                    try:
                        out = subprocess.check_output(['tail', '-30', str(log_path)],
                            stderr=subprocess.DEVNULL).decode()
                        for line in out.splitlines():
                            print(f'    | {line}')
                    except Exception as e:
                        print(f'    | (could not read log: {e})')
                    print('    --- end log ---')
                done_this_round.append(key)
        for key in done_this_round:
            del remaining[key]
        if remaining:
            poll_count += 1
            if poll_count % 3 == 0:
                snapshot(remaining, total, t0)
            time.sleep(30)
except KeyboardInterrupt:
    print('\n!!! Interrupted !!!')
    for key, (proc, logf, _, _) in remaining.items():
        try:
            proc.send_signal(signal.SIGKILL); logf.close()
        except Exception:
            pass
    raise

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
print(f'failures: {len(failures)}/{total}')
if failures:
    print('  failed keys:', failures)
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 7b. EMERGENCY kill.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'c3_phase3_exit_criterion.py' in line:
        try:
            pid = int(line.strip().split()[0])
            os.kill(pid, signal.SIGKILL); killed += 1
        except Exception as e:
            print(f'  failed to kill {line[:60]}: {e}')
print(f'killed {killed} c3 driver processes')

In [ ]:
# 8. Copy all per-seed result dirs + logs to Drive.
import shutil, os
dst_root = '/content/drive/MyDrive/neuro-ai/results/gamma1_f1_lr_cr_sweep_2026-05-27'
os.makedirs(dst_root, exist_ok=True)
LR_CR_VALUES = [0.01, 0.05, 0.1, 0.2, 0.5]
SEEDS = [0, 1, 2]
count = 0
for lr in LR_CR_VALUES:
    for seed in SEEDS:
        src = f'reports/f1_lr{lr}_seed{seed}_2026-05-27'
        if os.path.isdir(src):
            shutil.copytree(src, f'{dst_root}/lr{lr}_seed{seed}', dirs_exist_ok=True)
            count += 1
if os.path.isdir('reports/f1_logs'):
    shutil.copytree('reports/f1_logs', f'{dst_root}/colab_logs', dirs_exist_ok=True)
print(f'copied {count} per-seed dirs + logs to {dst_root}')

In [ ]:
# 9. AGGREGATION — lr_cr → mean Δ curve + hypothesis verdict.
#    Reads each lr_cr's n=3 per-seed JSONs, computes stratum-pooled
#    per-seed Δ (default mode, tight+spread), and the per-lr_cr mean.
#    PathC baseline at seeds 0..2 is loaded from the Γ1 headline gate's
#    Drive output for the paired comparison column.
import json
from pathlib import Path
from collections import defaultdict

LR_CR_VALUES = [0.01, 0.05, 0.1, 0.2, 0.5]
SEEDS = [0, 1, 2]
PRIMARY_STRATA = ['tight', 'spread']
DRIVE_F1 = '/content/drive/MyDrive/neuro-ai/results/gamma1_f1_lr_cr_sweep_2026-05-27'
DRIVE_PATHC = '/content/drive/MyDrive/neuro-ai/results/gamma1_headline_2026-05-27'

def extract_seed_counts(summary, mode, stratum, condition_key):
    try:
        cell = summary['aggregated'][mode][condition_key][stratum]
        return (int(cell['successes']), int(cell['trials']))
    except (KeyError, TypeError):
        return (0, 0)

def stratum_pooled_delta(summary, mode='default'):
    std_h = std_t = ctrl_h = ctrl_t = 0
    for stratum in PRIMARY_STRATA:
        sh, st = extract_seed_counts(summary, mode, stratum, 'standard')
        ch, ct = extract_seed_counts(summary, mode, stratum, 'shuffled_control')
        std_h += sh; std_t += st; ctrl_h += ch; ctrl_t += ct
    if std_t and ctrl_t:
        return std_h/std_t - ctrl_h/ctrl_t
    return None

# Load F1 per-seed deltas.
f1_deltas = defaultdict(dict)  # f1_deltas[lr][seed] = Δ
for lr in LR_CR_VALUES:
    for seed in SEEDS:
        local = Path(f'reports/f1_lr{lr}_seed{seed}_2026-05-27/c3_summary.json')
        drive = Path(f'{DRIVE_F1}/lr{lr}_seed{seed}/c3_summary.json')
        path = local if local.exists() else drive
        if path.exists():
            with path.open() as f:
                s = json.load(f)
            d = stratum_pooled_delta(s)
            if d is not None:
                f1_deltas[lr][seed] = d
        else:
            print(f'MISSING: {local} (and Drive fallback {drive})')

# Load PathC baseline at seeds 0..2 from headline gate's Drive output.
pathc_deltas = {}
for seed in SEEDS:
    p = Path(f'{DRIVE_PATHC}/pathc_baseline_seed{seed}/c3_summary.json')
    if p.exists():
        with p.open() as f:
            s = json.load(f)
        d = stratum_pooled_delta(s)
        if d is not None:
            pathc_deltas[seed] = d
    else:
        print(f'PathC baseline seed {seed} not found on Drive — paired comparison will be partial')

# Print per-lr_cr table.
print('=' * 80)
print('F1 — lr_cr → per-seed Δ (default mode, stratum-pooled tight+spread)')
print('=' * 80)
print(f'\n{"lr_cr":>8}  {"seed0":>8}  {"seed1":>8}  {"seed2":>8}  {"mean":>8}  {"min":>8}  {"max":>8}')
print('-' * 70)
summary_by_lr = {}
for lr in LR_CR_VALUES:
    vals = [f1_deltas[lr].get(seed) for seed in SEEDS]
    available = [v for v in vals if v is not None]
    if available:
        mean_v = sum(available) / len(available)
        min_v = min(available)
        max_v = max(available)
        summary_by_lr[lr] = {'mean': mean_v, 'min': min_v, 'max': max_v, 'n': len(available)}
        row = [f'{v:+.3f}' if v is not None else '   N/A' for v in vals]
        print(f'{lr:>8}  {row[0]:>8}  {row[1]:>8}  {row[2]:>8}  '
              f'{mean_v:+8.4f}  {min_v:+.3f}  {max_v:+.3f}')
    else:
        print(f'{lr:>8}  ALL MISSING')

# PathC reference at the matched seeds.
print('\n' + '-' * 70)
print('Reference: PathC pull/push baseline at matched seeds 0..2 (from headline gate)')
if pathc_deltas:
    pc_vals = [pathc_deltas.get(s) for s in SEEDS]
    pc_avail = [v for v in pc_vals if v is not None]
    pc_mean = sum(pc_avail) / len(pc_avail) if pc_avail else 0.0
    row = [f'{v:+.3f}' if v is not None else '   N/A' for v in pc_vals]
    print(f'   PathC  {row[0]:>8}  {row[1]:>8}  {row[2]:>8}  '
          f'{pc_mean:+8.4f}')
else:
    pc_mean = None
    print('   PathC: no baseline available on Drive — load skipped')

# Hypothesis verdict (per F1 precommit §"Falsifiable diagnostic criterion").
print('\n' + '=' * 80)
print('HYPOTHESIS VERDICT')
print('=' * 80)
if not summary_by_lr:
    print('  All F1 cells missing — cannot evaluate.')
else:
    max_mean = max(s['mean'] for s in summary_by_lr.values())
    argmax_lr = max(summary_by_lr.keys(), key=lambda k: summary_by_lr[k]['mean'])
    print(f'  Best lr_cr by per-seed mean Δ: lr_cr={argmax_lr}, mean Δ={max_mean:+.4f}')
    if pc_mean is not None:
        print(f'  PathC reference mean Δ (seeds 0..2): {pc_mean:+.4f}')
    print()
    if max_mean >= 0.04:
        print('  ⇒ HYPOTHESIS (a): effective-lr mismatch.')
        print(f'     Best Γ1.c per-seed mean Δ ({max_mean:+.4f}) is within ~30% of PathC')
        print(f'     ({pc_mean:+.4f}). The shape change CAN produce PathC-magnitude effects')
        print(f'     at the right lr_cr.')
        print(f'  NEXT: draft a new precommit for a full n=10 headline gate at')
        print(f'        lr_cr={argmax_lr}. Same H1-H7 inheritance; same operating point;')
        print(f'        anti-homunculus reviewer NOT required again (mechanism unchanged).')
    elif max_mean <= 0.02:
        print('  ⇒ HYPOTHESIS (b): atom-vs-atom geometry carries less corpus signal.')
        print(f'     Best Γ1.c per-seed mean Δ ({max_mean:+.4f}) is ≲ 0.02 across all')
        print(f'     lr_cr values. The shape itself does not carry PathC-magnitude signal,')
        print(f'     regardless of magnitude.')
        print(f'  NEXT: close the Γ1 family. Draft Γ2 (bundle-first scene memory)')
        print(f'        precommit per path-gamma-mechanism-family-survey.md, with full')
        print(f'        mp-grill-with-docs + anti-homunculus reviewer + auditor cycle.')
    else:
        print('  ⇒ INCONCLUSIVE.')
        print(f'     Best Γ1.c per-seed mean Δ ({max_mean:+.4f}) is between 0.02 and 0.04')
        print(f'     — promising but not clearly distinguishing the two hypotheses.')
        print(f'  NEXT: extend F1 at the most promising lr_cr (={argmax_lr}) to n=5 or')
        print(f'        n=10. Still diagnostic, not graduation.')

# Save aggregate JSON to Drive.
import os
agg = {
    'tag': 'gamma1_f1_lr_cr_sweep_2026-05-27',
    'precommit': 'notes/notes/2026-05-27-path-gamma-gamma1-f1-lr-cr-sweep-precommit.md',
    'parent_precommit': 'notes/notes/2026-05-27-path-gamma-gamma1-context-residual-precommit.md',
    'parent_report': 'reports/113_path_gamma_gamma1_headline_gate.md',
    'per_lr_summary': {
        str(lr): summary_by_lr.get(lr, {}) for lr in LR_CR_VALUES
    },
    'per_seed_deltas_f1': {
        str(lr): {str(s): float(f1_deltas[lr][s]) for s in f1_deltas[lr]}
        for lr in LR_CR_VALUES
    },
    'pathc_per_seed_deltas': {str(k): float(v) for k, v in pathc_deltas.items()},
    'pathc_reference_mean': float(pc_mean) if pc_mean is not None else None,
    'best_lr_cr': argmax_lr if summary_by_lr else None,
    'best_mean_delta': max_mean if summary_by_lr else None,
}
agg_path = f'{DRIVE_F1}/aggregate.json'
os.makedirs(os.path.dirname(agg_path), exist_ok=True)
with open(agg_path, 'w') as f:
    json.dump(agg, f, indent=2)
print(f'\nwrote aggregate to {agg_path}')